# India National Power Grid Telemetry - Heavy Compute Training Pipeline

This notebook implements the complete offline heavy-compute training pipeline for the Grid Telemetry Engine:
1. Ingestion & Polars feature engineering with zero lookahead leakage
2. Multi-horizon LightGBM quantile regression (p10, p50, p90) per region
3. Unsupervised Isolation Forest calibration
4. SHAP k-means background centroids and Evidently AI baseline export
5. Automated artifact zip packaging and download link generation

In [ ]:
# [cell 1] install dependencies and prepare workspace
!pip install -q \
    "polars>=0.20.0" \
    "lightgbm>=4.3.0" \
    "shap>=0.45.0" \
    "evidently>=0.4.15" \
    "holidays>=0.45" \
    "joblib>=1.3.2" \
    "pydantic>=2.6.0" \
    "scikit-learn>=1.4.0"

import os
import json
import shutil
import hashlib
import zipfile
import numpy as np
import polars as pl
import lightgbm as lgb
import joblib
from datetime import datetime, timezone
from IPython.display import FileLink, display

ARTIFACTS_DIR = "/kaggle/working/artifacts"
MODELS_DIR = os.path.join(ARTIFACTS_DIR, "models")
BASELINES_DIR = os.path.join(ARTIFACTS_DIR, "baselines")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(BASELINES_DIR, exist_ok=True)
print(f"Directory initialized: {ARTIFACTS_DIR}")

In [ ]:
# [cell 2] feature engineering with polars
import holidays

def get_indian_holidays(years: list[int]) -> set:
    ind_holidays = holidays.India(years=years)
    return set(ind_holidays.keys())

def build_features(df_raw: pl.DataFrame) -> pl.DataFrame:
    """
    Transforms raw telemetry into a zero-leakage feature matrix.
    Guaranteed deterministic across historical and live streaming batches.
    """
    years = df_raw["timestamp_utc"].dt.year().unique().to_list()
    holidays_set = get_indian_holidays(years)
    
    lazy_df = (
        df_raw.lazy()
        .sort(["region", "timestamp_utc"])
        .with_columns([
            pl.col("timestamp_utc").dt.hour().alias("hour"),
            pl.col("timestamp_utc").dt.weekday().alias("day_of_week"),
            pl.col("timestamp_utc").dt.ordinal_day().alias("day_of_year"),
            pl.col("timestamp_utc").dt.date().is_in(holidays_set).cast(pl.Int8).alias("is_holiday")
        ])
        .with_columns([
            (2 * np.pi * pl.col("hour") / 24.0).sin().alias("sin_hour_k1"),
            (2 * np.pi * pl.col("hour") / 24.0).cos().alias("cos_hour_k1"),
            (4 * np.pi * pl.col("hour") / 24.0).sin().alias("sin_hour_k2"),
            (4 * np.pi * pl.col("hour") / 24.0).cos().alias("cos_hour_k2"),
            (2 * np.pi * pl.col("day_of_year") / 365.25).sin().alias("sin_doy"),
            (2 * np.pi * pl.col("day_of_year") / 365.25).cos().alias("cos_doy"),
        ])
        .with_columns([
            pl.col("demand_mw").shift(lag).over("region").alias(f"demand_lag_{lag}h")
            for lag in [1, 2, 3, 24, 48, 168]
        ])
        .with_columns([
            pl.col("demand_mw").shift(1).rolling_mean(window_size=w).over("region").alias(f"demand_rolling_mean_{w}h")
            for w in [6, 24, 168]
        ] + [
            pl.col("demand_mw").shift(1).rolling_std(window_size=w).over("region").alias(f"demand_rolling_std_{w}h")
            for w in [6, 24, 168]
        ])
        .with_columns([
            (pl.col("temperature_2m") - pl.col("temperature_2m").shift(24).over("region")).alias("temp_delta_24h"),
            (pl.col("relative_humidity_2m") - pl.col("relative_humidity_2m").shift(24).over("region")).alias("rh_delta_24h")
        ])
        .drop_nulls(subset=["demand_lag_168h", "demand_rolling_std_168h"])
    )
    return lazy_df.collect()

In [ ]:
# [cell 3] quantile lightgbm model training
FEATURE_COLUMNS = [
    "sin_hour_k1", "cos_hour_k1", "sin_hour_k2", "cos_hour_k2",
    "sin_doy", "cos_doy", "day_of_week", "is_holiday",
    "demand_lag_1h", "demand_lag_2h", "demand_lag_3h",
    "demand_lag_24h", "demand_lag_48h", "demand_lag_168h",
    "demand_rolling_mean_6h", "demand_rolling_mean_24h", "demand_rolling_mean_168h",
    "demand_rolling_std_6h", "demand_rolling_std_24h", "demand_rolling_std_168h"
]
TARGET = "demand_mw"
QUANTILES = [0.10, 0.50, 0.90]
REGIONS = ["ALL_INDIA", "NR", "WR", "SR", "ER", "NER"]

def train_regional_quantiles(df_features: pl.DataFrame):
    trained_catalog = {}
    for region in REGIONS:
        print(f"Training LightGBM models: {region}")
        df_region = df_features.filter(pl.col("region") == region).to_pandas()
        X = df_region[FEATURE_COLUMNS]
        y = df_region[TARGET]
        
        split_idx = int(len(df_region) * 0.90)
        X_train, y_train = X.iloc[:split_idx], y.iloc[:split_idx]
        X_val, y_val = X.iloc[split_idx:], y.iloc[split_idx:]
        
        region_models = {}
        for q in QUANTILES:
            params = {
                "objective": "quantile",
                "alpha": q,
                "boosting_type": "gbdt",
                "n_estimators": 1000,
                "learning_rate": 0.03,
                "num_leaves": 45,
                "max_depth": 7,
                "subsample": 0.8,
                "colsample_bytree": 0.8,
                "verbosity": -1,
                "n_jobs": -1
            }
            dtrain = lgb.Dataset(X_train, label=y_train)
            dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
            booster = lgb.train(
                params,
                dtrain,
                valid_sets=[dval],
                callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
            )
            model_file = f"lgb_quantile_{int(q * 100)}_{region}.txt"
            booster.save_model(os.path.join(MODELS_DIR, model_file))
            region_models[str(q)] = model_file
            
        trained_catalog[region] = region_models
        
    with open(os.path.join(MODELS_DIR, "model_registry.json"), "w") as f:
        json.dump(trained_catalog, f, indent=2)
    with open(os.path.join(ARTIFACTS_DIR, "feature_columns.json"), "w") as f:
        json.dump(FEATURE_COLUMNS, f, indent=2)

In [ ]:
# [cell 4] unsupervised anomaly engine calibration
from sklearn.ensemble import IsolationForest

def train_isolation_forests(df_features: pl.DataFrame):
    for region in REGIONS:
        df_region = df_features.filter(pl.col("region") == region).to_pandas()
        X = df_region[FEATURE_COLUMNS]
        iso = IsolationForest(
            n_estimators=300,
            contamination=0.01,
            random_state=42,
            n_jobs=-1
        )
        iso.fit(X)
        joblib.dump(iso, os.path.join(MODELS_DIR, f"isolation_forest_{region}.joblib"), compress=3)
        print(f"Calibrated Isolation Forest: {region}")

In [ ]:
# [cell 5] shap background and evidently reference window export
import shap

def export_baselines(df_features: pl.DataFrame):
    for region in REGIONS:
        df_region = df_features.filter(pl.col("region") == region).to_pandas()
        
        # 1. evidently reference data: last 30 days (720 hourly records)
        ref_df = df_region.iloc[-720:][FEATURE_COLUMNS + [TARGET]]
        ref_path = os.path.join(BASELINES_DIR, f"reference_baseline_{region}.parquet")
        ref_df.to_parquet(ref_path, index=False)
        
        # 2. shap background set: 150 k-means cluster centers
        sample_size = min(3000, len(df_region))
        X_sample = df_region[FEATURE_COLUMNS].sample(n=sample_size, random_state=42)
        shap_bg = shap.kmeans(X_sample, k=150)
        joblib.dump(shap_bg, os.path.join(BASELINES_DIR, f"shap_background_{region}.joblib"), compress=3)
        print(f"Exported baselines for {region}")

In [ ]:
# [cell 6] automated packaging and one-click zip generation
OUTPUT_ZIP = "/kaggle/working/grid_engine_artifacts.zip"

def package_artifacts():
    print("Packaging trained models, schemas, and baselines...")
    with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(ARTIFACTS_DIR):
            for file in files:
                abs_path = os.path.join(root, file)
                rel_path = os.path.relpath(abs_path, start="/kaggle/working")
                zipf.write(abs_path, rel_path)
                hasher = hashlib.md5()
                with open(abs_path, "rb") as f:
                    hasher.update(f.read())
                print(f" + Added {rel_path} [MD5: {hasher.hexdigest()[:8]}]")
                
    size_mb = os.path.getsize(OUTPUT_ZIP) / (1024 * 1024)
    print(f"\nSuccessfully created bundle: {OUTPUT_ZIP} ({size_mb:.2f} MB)")
    print("Click link below to download artifact bundle:\n")
    display(FileLink("grid_engine_artifacts.zip"))

package_artifacts()